# Vectorless RAG — BM25 Walkthrough

This notebook walks through every step of the pipeline one cell at a time.

| Step | Description |
|---|---|
| 1 | Load documents (PDF + .txt) |
| 2 | Split into chunks |
| 3 | Build a BM25 index |
| 4 | Query the index |
| 5 | Generate an answer with Ollama |

No embeddings. No vector database. Just BM25.

## 0. Setup

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# Edit these to point at your documents and preferred model

PDF_DIR      = "../data/pdf_files"    # ← your PDFs
TEXT_DIR     = "../data/text_files"   # ← your .txt files
OLLAMA_MODEL = "llama3.2"             # ← any model from `ollama list`
CHUNK_SIZE   = 1000
CHUNK_OVERLAP = 200
TOP_K        = 5
# ─────────────────────────────────────────────────────────────────────────────

from pathlib import Path
print(f"PDF dir  : {Path(PDF_DIR).resolve()}")
print(f"Text dir : {Path(TEXT_DIR).resolve()}")
print(f"Model    : {OLLAMA_MODEL}")

## 1. Load Documents

In [ ]:
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_core.documents import Document

docs: list[Document] = []

# PDFs
for pdf_file in sorted(Path(PDF_DIR).glob("*.pdf")):
    print(f"  Loading PDF : {pdf_file.name}")
    loader = PyPDFLoader(str(pdf_file))
    pages = loader.load()
    for page in pages:
        page.metadata.update({"source_file": pdf_file.name, "file_type": "pdf"})
    docs.extend(pages)

# Text files
for txt_file in sorted(Path(TEXT_DIR).glob("*.txt")):
    print(f"  Loading TXT : {txt_file.name}")
    loader = TextLoader(str(txt_file), encoding="utf-8")
    pages = loader.load()
    for page in pages:
        page.metadata.update({"source_file": txt_file.name, "file_type": "text"})
    docs.extend(pages)

print(f"\nTotal pages loaded: {len(docs)}")
print(f"\nSample metadata: {docs[0].metadata if docs else 'no docs'}")

## 2. Split into Chunks

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", " ", ""],
)
chunks = splitter.split_documents(docs)

print(f"Total chunks: {len(chunks)}")
print(f"\nFirst chunk (first 300 chars):")
print(chunks[0].page_content[:300])
print(f"\nMetadata: {chunks[0].metadata}")

## 3. Build the BM25 Index

BM25 (Best Match 25) is a probabilistic ranking function that scores documents
based on how often query terms appear, weighted by their rarity across the corpus.

- **TF (Term Frequency)**: How often does the term appear in this chunk?
- **IDF (Inverse Document Frequency)**: How rare is this term across all chunks?
- **Length normalisation**: Longer chunks are not unfairly rewarded.

In [ ]:
from rank_bm25 import BM25Okapi

def tokenise(text: str) -> list[str]:
    return text.lower().split()

# Build the corpus and index
corpus = [{"text": c.page_content, "metadata": c.metadata} for c in chunks]
tokenised_corpus = [tokenise(c.page_content) for c in chunks]
bm25 = BM25Okapi(tokenised_corpus)

print(f"BM25 index built over {len(corpus)} chunks.")
print(f"Vocabulary size: {len(bm25.idf)} unique tokens")

## 4. Query the Index

In [ ]:
# ── Try your own question here ─────────────────────────────────────────────────
QUESTION = "What are the daily cleaning steps for the coffee machine?"
# ──────────────────────────────────────────────────────────────────────────────

import numpy as np

query_tokens = tokenise(QUESTION)
scores = bm25.get_scores(query_tokens)
ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)[:TOP_K]

results = [
    {"document": corpus[i]["text"], "metadata": corpus[i]["metadata"], "score": float(s)}
    for i, s in ranked
]

print(f"Question : {QUESTION}\n")
print(f"Top {TOP_K} chunks:")
for i, r in enumerate(results, 1):
    src     = r["metadata"].get("source_file", "?")
    pg      = r["metadata"].get("page", "?")
    snippet = r["document"][:120].replace("\n", " ").strip()
    print(f"  {i}. [score {r['score']:.2f}] {src} p.{pg}")
    print(f"     {snippet}…")
    print()

## 5. Generate an Answer with Ollama

In [ ]:
import ollama

context_parts = []
for r in results:
    source = r["metadata"].get("source_file", "unknown")
    page   = r["metadata"].get("page", "?")
    context_parts.append(f"[Source: {source}, page {page}]\n{r['document']}")

context = "\n\n---\n\n".join(context_parts)

prompt = (
    "You are a helpful assistant. Answer the question using only the context below.\n"
    'If the answer is not in the context, say: "I don\'t have enough information to answer that."\n\n'
    f"Context:\n{context}\n\n"
    f"Question: {QUESTION}\n\n"
    "Answer:"
)

print(f"Sending {len(results)} chunks to {OLLAMA_MODEL} …\n")
response = ollama.chat(
    model=OLLAMA_MODEL,
    messages=[{"role": "user", "content": prompt}],
)
print("Answer:")
print(response["message"]["content"])

## 6. Compare BM25 Scores Across Questions

Run this cell to see how BM25 scores differ for keyword-rich vs vague queries.

In [ ]:
test_questions = [
    "What are the daily cleaning steps?",              # keyword-rich
    "How do I descale the machine?",                   # specific procedure
    "Why does my coffee taste bitter?",                # troubleshooting
    "What temperature should I use for light roasts?", # settings
    "Tell me something about the machine",             # vague
]

print(f"{'Question':<50}  {'Top score':>10}  Top source")
print("-" * 90)

for q in test_questions:
    q_scores = bm25.get_scores(tokenise(q))
    best_idx  = int(q_scores.argmax())
    best_score = q_scores[best_idx]
    best_src   = corpus[best_idx]["metadata"].get("source_file", "?")
    print(f"{q:<50}  {best_score:>10.2f}  {best_src}")